# COMPOSITION

In [1]:
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda
)

In [16]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()
MODEL = os.getenv("MODEL")
MODEL_PROVIDER= os.getenv("MODEL_PROVIDER")

model = init_chat_model(model=MODEL, model_provider=MODEL_PROVIDER, temperature=0.6, reasoning=False)

prompt= ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Your answer length will be {n}'),
    ('human', '{question}'),
])

parser = StrOutputParser()

In [ ]:
chain = (

    RunnableParallel(
        {"question": RunnablePassthrough(),
     "n": RunnableLambda(lambda x: len(x)*2)}

    )
    | prompt
    | model
    | parser

)
# for example, getting different perspectives from different domain specialists -> RunnableParallel

In [ ]:
chain.invoke('What kind of music was played in 1600s England?')

"Hello! How can I assist you today? Feel free to ask me any questions or let me know if you need help with something specific. I'm here to help! 😊"

In [ ]:
from langchain_core.runnables import RunnableLambda

upper = RunnableLambda(lambda x: x['q'].upper())

result = upper.invoke({'q':'hello'})

print(result) # str

HELLO


In [14]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

parallel = RunnableParallel({

    "len": RunnableLambda(lambda x: len(x['q'])),
    "echo": RunnablePassthrough(),

})

result = parallel.invoke({'q':'hello'})

print(result) # dict

{'len': 5, 'echo': {'q': 'hello'}}


In [17]:
from langchain_core.runnables import RunnablePassthrough

base = RunnablePassthrough()

chain = base.assign(
    summary = prompt | model | parser
)

result = chain.invoke({
    "question": "what is LCEL?",
    "n": 10,
})

print(result.keys())
print("question: ", result["question"])
print("n: ", result['n'])
print("summary: ", result['summary'])

dict_keys(['question', 'n', 'summary'])
question:  what is LCEL?
n:  10
summary:  LCEL is LangChain Expression Language.


In [21]:
from langchain_core.runnables import RunnableLambda

broken_model = init_chat_model(
    model="qwen2.5:27b",
    model_provider=MODEL_PROVIDER,
    temperature=.9,
    validate_model_on_init=False
)

primary= broken_model | parser

backup = RunnableLambda(lambda x : "FALLBACK: model failed, using backup answer.")

chain = primary.with_fallbacks([backup])

result = chain.invoke("hello")

print(result)

FALLBACK: model failed, using backup answer.


In [22]:
primary = RunnableLambda(lambda x: 1 / 0)  # ZeroDivisionError
backup = RunnableLambda(lambda x: f"backup got: {x}")

chain = primary.with_fallbacks([backup])
print(chain.invoke("test"))  # backup got: test

backup got: test
